In [ ]:
# ===== ROLL C: RFM segmenteerimine =====

# 1. Viitekuupäev
today = df["sale_date"].max() + pd.Timedelta(days=1)

# 2. Recency — mitu päeva on möödunud viimasest ostust
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

# 3. Frequency — ostude arv
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

# 4. Monetary — kogukulutus
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

# 5. Ühenda RFM tabel
rfm = recency.merge(frequency, on='customer_id').merge(monetary, on='customer_id')

# 6. Skoorid 1-5
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# 7. Segmendid
def segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7:  return 'Potential'
    elif score >= 4:  return 'At Risk'
    else:             return 'Lost'

rfm['segment'] = rfm['RFM_Score'].apply(segment)

# 8. Kokkuvõte
summary = rfm['segment'].value_counts().reset_index()
summary.columns = ['segment', 'arv']
summary['osakaal_%'] = (summary['arv'] / len(rfm) * 100).round(1)
print(summary)